# pytorch quantization

shrinking models for inference. dynamic vs static.

In [ ]:
import torch
import torch.nn as nn
import torch.quantization

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(128, 64)
        self.l2 = nn.Linear(64, 10)
    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))

m = Net().eval()

q = torch.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
x = torch.randn(1, 128)
torch.allclose(m(x), q(x), atol=1e-1)

## size comparison

In [ ]:
import os
torch.save(m.state_dict(), '/tmp/m.pt')
torch.save(q.state_dict(), '/tmp/q.pt')
print('fp32:', os.path.getsize('/tmp/m.pt'))
print('int8:', os.path.getsize('/tmp/q.pt'))

## static quant requires calibration
```python
m.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(m, inplace=True)
# run calib batches through m here
torch.quantization.convert(m, inplace=True)
```

tested on a small lstm: ~3.8x size reduction, marginal accuracy hit. worth it for cpu inference.

small fix to comment.

saving for later.

smaller batch helped on this one. counterintuitive.

tried bumping batch size, no real change.